# 02 — Pipeline CPU n°2 : PyTorch → TorchScript / torch.compile → CPU

| Couche | Technologie |
|---|---|
| Haut niveau | PyTorch, ResNet-18 + tête 10 classes |
| Format du modèle | TorchScript (graphe figé) / graphe PyTorch 2 + Inductor |
| Middle level | Compilation et optimisation du graphe PyTorch |
| Bas niveau | Kernels CPU générés/sélectionnés par PyTorch |
| Matériel | CPU |

Ce notebook mesure deux variantes natives de PyTorch : TorchScript (tracé +
gel + optimisation pour l'inférence) et `torch.compile` (Inductor, génération
de kernels CPU).

In [ ]:
%matplotlib inline
import json
import os
import sys
import time
from pathlib import Path

def _find_root() -> Path:
    p = Path.cwd().resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "src" / "common.py").exists():
            return candidate
    return p

ROOT = _find_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ.setdefault("TP_ROOT", str(ROOT))
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")

IN_COLAB = "google.colab" in sys.modules
REPO_URL = os.environ.get("TP_REPO_URL", "")
if IN_COLAB and REPO_URL and not (ROOT / "src" / "common.py").exists():
    import subprocess
    subprocess.run(["git", "clone", REPO_URL], check=True)
    ROOT = Path(REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")).resolve()
    os.chdir(ROOT)
    sys.path.insert(0, str(ROOT))
    os.environ["TP_ROOT"] = str(ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown
from src import common as C

C.ensure_dirs()
C.set_seed()
torch.set_num_threads(int(os.environ.get("OMP_NUM_THREADS", "4")))
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
print("Racine du projet :", ROOT)
print("PyTorch :", torch.__version__, "| CUDA :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
print(f"Protocole : {C.N_IMAGES} images | batch={C.BATCH_SIZE} | {C.REPETITIONS} répétitions | warmup={C.WARMUP}")

In [ ]:
def show(result_or_id):
    result = C.load_result(result_or_id) if isinstance(result_or_id, str) else result_or_id
    print(f"{result['label']}  |  device={result['device']}  |  backend={result['backend']}")
    display(pd.DataFrame(C.summary_rows(result)).round(4))
    print(f"Top-1 accuracy = {result['accuracy_top1'] * 100:.2f} %")

def bar_metric(ids, metric, std_metric, ylabel, title, color="#4C72B0"):
    df = pd.DataFrame(C.comparison_rows(ids))
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(df["pipeline"], df[metric], yerr=df[std_metric], capsize=5, color=color)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=18)
    for i, value in enumerate(df[metric]):
        if value is not None:
            ax.text(i, value, f"{value:.2f}", ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    plt.show()
    return df

## 1. Données et modèle

In [ ]:
images, labels = C.load_subset()
x_cpu = C.preprocess_images(images, device="cpu")
print("images :", images.shape, images.dtype)
print("tenseur prétraité :", tuple(x_cpu.shape), x_cpu.dtype)
print("distribution :", np.bincount(labels, minlength=len(C.CLASSES)).tolist())
print("classes :", C.CLASSES)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, idx in zip(axes.flat, range(10)):
    ax.imshow(images[idx])
    ax.set_title(C.CLASSES[int(labels[idx])])
    ax.axis("off")
fig.suptitle("Exemples du sous-ensemble CIFAR-10 (5 premières classes)")
plt.tight_layout()
plt.show()

In [ ]:
model = C.build_model("cpu")
n_params = sum(p.numel() for p in model.parameters())
print(f"ResNet-18 + tête 10 classes : {n_params/1e6:.2f} M paramètres")
print("Couche de classification :", model.fc)
print("Poids de la tête :", C.ARTIFACTS / "classifier_head.npz")

## 2. TorchScript

In [ ]:
example = torch.randn(C.BATCH_SIZE, 3, C.IMAGE_SIZE, C.IMAGE_SIZE)
traced = torch.jit.trace(model, example)
traced = torch.jit.freeze(traced.eval())
traced = torch.jit.optimize_for_inference(traced)
ts_path = C.ARTIFACTS / "resnet18_cifar10_torchscript_cpu.pt"
torch.jit.save(traced, ts_path)
print("TorchScript :", ts_path, f"({ts_path.stat().st_size/1e6:.2f} Mo)")

fn_ts = C.make_torch_fn(traced, "cpu")
records, summary, logits_ts = C.run_benchmark(fn_ts, x_cpu, "cpu")
C.save_result(
    "02a_cpu_torchscript", records, summary, logits_ts, labels,
    device="cpu", backend="torchscript",
    notes="TorchScript tracé, gelé et optimisé pour l'inférence.",
)
show("02a_cpu_torchscript")

## 3. torch.compile (Inductor)

`torch.compile` capture le graphe, le compile avec Inductor et génère des
kernels C++/OpenMP pour le CPU. La première exécution compile ; les warmups
absorbent ce coût avant les mesures.

In [ ]:
compiled = torch.compile(model, mode="default", fullgraph=False, dynamic=False)
fn_compiled = C.make_torch_fn(compiled, "cpu")
records, summary, logits_compiled = C.run_benchmark(fn_compiled, x_cpu, "cpu")
C.save_result(
    "02_cpu_torchscript_compile", records, summary, logits_compiled, labels,
    device="cpu", backend="torch.compile/inductor",
    notes="torch.compile mode=default, Inductor CPU, 4 threads.",
)
show("02_cpu_torchscript_compile")

## 4. Comparaison

In [ ]:
ref = np.load(C.RESULTS / "00_reference_pytorch_cpu_logits.npy")
for rid in ["02a_cpu_torchscript", "02_cpu_torchscript_compile"]:
    r = C.load_result(rid)
    logits_r = np.load(C.RESULTS / f"{rid}_logits.npy")
    print(
        f"{r['label']:<34} Top-1={r['accuracy_top1']*100:5.2f} %  "
        f"accord={C.prediction_agreement(logits_r, ref)*100:5.2f} %"
    )
ids = ["00_reference_pytorch_cpu", "02a_cpu_torchscript", "02_cpu_torchscript_compile"]
display(pd.DataFrame(C.comparison_rows(ids)).round(4))
bar_metric(ids, "throughput_ips", "throughput_std", "images/s",
           "Pipeline CPU n°2 : débit", color="#8172B2")
bar_metric(ids, "latency_ms", "latency_std", "ms / image",
           "Pipeline CPU n°2 : latence", color="#CCB974")

### Interprétation

TorchScript fige le graphe et permet des optimisations statiques, tandis que
`torch.compile` va plus loin en générant des kernels spécialisés. Le gain dépend
fortement du CPU, du nombre de threads et du coût de compilation. Les warmups
sont exclus des mesures : on mesure le régime établi, pas la compilation.